# Reactor Scale and Cell Type Library
Sample realistic stirred-tank bioreactors at different scales, and realistic CHO cell types, for simulations and benchmarks.

- **Reactors** are drawn from ranges typical of commercial vessels (Ambr 15 and 250, glass bench-top, pilot, 200 L and production tanks). The geometry is derived consistently from the volume, so oxygen transfer (van't Riet, 1979), mixing time (Grenville; Nienow, 1997) and shear (Nienow, 2006) follow from physics.
- **Cell types** are drawn around a calibrated CHO base cell line, within ranges reported for CHO fed-batch processes.

The ranges live in `dbiolab.mammalian_cellculture_simulation.scale_library` (`REACTOR_SCALES`, `CELLTYPE_RANGES`); edit them there.

In [ ]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

from dbiolab.mammalian_cellculture_simulation import (
    REACTOR_SCALES,
    FeedConcentrations,
    FeedingEvent,
    MammalianCellcultureReactor,
    MammalianRecipe,
    NoiseParameters,
    sample_celltypes,
    sample_reactor,
    stirring_range,
)
from dbiolab.mammalian_cellculture_simulation.scale_library import CELLTYPE_RANGES

rng = np.random.default_rng(seed=42)

## 1. Reactors at Each Scale
One sampled reactor per scale. The stirring range is the one giving the typical cell-culture power input of 5-100 W/m3.

In [ ]:
reactors = {scale: sample_reactor(scale, rng) for scale in REACTOR_SCALES}

print(
    f"{'scale':<11}{'volume':>10}{'tank D':>8}{'impeller D':>11}"
    f"{'stirring':>15}{'kLa':>15}{'mixing time':>14}"
)
print(f"{'':<11}{'[L]':>10}{'[m]':>8}{'[m]':>11}{'[rpm]':>15}{'[1/h]':>15}{'[s]':>14}")
for scale, reactor in reactors.items():
    low, high = stirring_range(reactor)
    print(
        f"{scale:<11}{reactor.volume:>10.3g}{reactor.tank_diameter:>8.3f}"
        f"{reactor.impeller_diameter:>11.3f}{low:>8.0f}-{high:<6.0f}"
        f"{reactor.kla(low):>7.1f}-{reactor.kla(high):<7.1f}"
        f"{reactor.mixing_time(low):>7.1f}-{reactor.mixing_time(high):<6.1f}"
    )

## 2. Engineering Characteristics Across Scales
At the same power input per volume, oxygen transfer is similar at every scale (it even rises at large scale with the taller liquid column), but large tanks mix much more slowly. Mixing times of tens of seconds create gradients of feed, base and oxygen that cost growth and viability. Large tanks also run at lower gas flow per volume, so their bubbles saturate with CO2 and strip it poorly.

In [ ]:
power_per_volume = np.geomspace(5.0, 100.0, 30)  # W/m3
kla_figure, mixing_figure = go.Figure(), go.Figure()
for scale, reactor in reactors.items():
    rpm = [stirring_range(reactor, (p, p))[0] for p in power_per_volume]
    kla_figure.add_trace(
        go.Scatter(x=power_per_volume, y=[reactor.kla(n) for n in rpm], name=scale)
    )
    mixing_figure.add_trace(
        go.Scatter(
            x=power_per_volume, y=[reactor.mixing_time(n) for n in rpm], name=scale
        )
    )
kla_figure.update_layout(
    title="Oxygen Transfer vs. Power Input",
    xaxis_title="P/V (W/m3)",
    yaxis_title="kLa (1/h)",
    xaxis_type="log",
    width=800,
    height=450,
)
mixing_figure.update_layout(
    title="Mixing Time vs. Power Input",
    xaxis_title="P/V (W/m3)",
    yaxis_title="Mixing time (s)",
    xaxis_type="log",
    yaxis_type="log",
    width=800,
    height=450,
)
kla_figure.show()
mixing_figure.show()

## 3. Cell Types
Sampled CHO cell lines. Set `N_CELLTYPES` to how many you want: Latin hypercube sampling splits every range in `CELLTYPE_RANGES` into that many equal slices and uses each slice once, so the cell types cover the ranges evenly (log-scaled parameters evenly in log space). The other parameters are copied from the base.

In [ ]:
N_CELLTYPES = 5  # how many cell types to sample
celltypes = sample_celltypes(N_CELLTYPES, seed=42)

shown = ["mu_max", "e_activation", "ph_opt", "t_opt_product", "ph_opt_product", "q_o2"]
print(f"{'name':<8}" + "".join(f"{key:>16}" for key in shown))
for celltype in celltypes:
    print(
        f"{celltype.name:<8}"
        + "".join(f"{getattr(celltype, key):>16.4g}" for key in shown)
    )
print("\nDoubling times [h]:", [round(np.log(2) / c.mu_max, 1) for c in celltypes])
print("Sampled parameters:", list(CELLTYPE_RANGES))

## 4. Save the Library
Writes every reactor and cell type as JSON, to load with `ReactorType.from_json` and `CellType.from_json`.

In [ ]:
library = Path("generated_library")
(library / "reactors").mkdir(parents=True, exist_ok=True)
(library / "celltypes").mkdir(parents=True, exist_ok=True)
for scale, reactor in reactors.items():
    reactor.to_json(str(library / "reactors" / f"{scale}.json"))
for celltype in celltypes:
    celltype.to_json(str(library / "celltypes" / f"{celltype.name}.json"))
sorted(str(p) for p in library.rglob("*.json"))

## 5. The Same Process at Every Scale
The tutorial's best recipe, scaled up the common way: constant power input per volume (30 W/m3), so each tank gets its own stirring rate. Dissolved oxygen is controlled at 40 % air saturation. Large tanks lose product to slower mixing and, with their lower gas flow per volume, to CO2 build-up.

In [ ]:
def make_feed(time_point, c_glc, c_gln, temperature, ph_value, stirring_rate):
    return FeedingEvent(
        time_point=time_point,
        feed_concentrations=FeedConcentrations(
            c_product=0.0,
            x_total=0.0,
            x_viable=0.0,
            x_dead=0.0,
            c_glc=c_glc,
            c_gln=c_gln,
            c_lac=0.0,
            c_ammonia=0.0,
            c_aggregates=0.0,
            c_glycated=0.0,
            c_hcp=0.0,
            c_o2=0.0,
            c_co2=0.0,
        ),
        volume_fraction=c_glc / 1000.0 + c_gln / 200.0,
        temperature=temperature,
        ph_value=ph_value,
        stirring_rate=stirring_rate,
    )


def scaled_recipe(reactor, celltype, power_per_volume=30.0):
    rpm = stirring_range(reactor, (power_per_volume, power_per_volume))[0]
    return MammalianRecipe(
        name=f"{reactor.name} / {celltype.name}",
        celltype=celltype,
        temperature=36.13,
        ph_value=6.89,
        stirring_rate=rpm,
        initial_conditions=FeedConcentrations(
            c_product=0.0,
            x_total=800000.0,
            x_viable=800000.0,
            x_dead=0.0,
            c_glc=5.02,
            c_gln=1.27,
            c_lac=0.0,
            c_ammonia=0.0,
            c_aggregates=0.0,
            c_glycated=0.0,
            c_hcp=0.0,
            c_o2=reactor.o2_saturation,
            c_co2=1.228,  # 40 mmHg
        ),
        feeding_strategy=[
            make_feed(72.0, 1.89, 1.1, 35.68, 6.86, rpm),
            make_feed(144.0, 12.39, 4.71, 34.4, 6.85, rpm),
            make_feed(216.0, 20.0, 4.97, 33.11, 6.92, rpm),
        ],
        runtime=336.0,
        measurement_intervals=12.0,
        noise_parameters=NoiseParameters(),
        do_setpoint=0.4,
    )


celltype = celltypes[0]
product_figure = go.Figure()
for scale, reactor in reactors.items():
    recipe = scaled_recipe(reactor, celltype)
    result = MammalianCellcultureReactor(reactor).run_experiment(recipe)
    product_figure.add_trace(
        go.Scatter(x=result.timepoints, y=result.c_product, name=scale)
    )
    print(
        f"{scale:<11} {recipe.stirring_rate:6.0f} rpm, "
        f"mixing {reactor.mixing_time(recipe.stirring_rate):5.1f} s: "
        f"product {result.c_product[-1]:.4f} mM, "
        f"peak pCO2 after day 2 "
        f"{(result.c_co2[result.timepoints > 48] / 0.0307).max():4.0f} mmHg"
    )
product_figure.update_layout(
    title=f"Product Across Scales ({celltype.name}, constant P/V)",
    xaxis_title="Time (hours)",
    yaxis_title="Product (mM)",
    width=800,
    height=450,
)
product_figure.show()